In [1]:
import torch
import logging
import os
from pathlib import Path
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, TrainerCallback
from datasets import load_dataset
from perf_estimator.trainer.plugins import ProfilerCallback, SnapshotCallback


/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"


In [3]:

# Configure logging
logging.basicConfig(format='%(asctime)s - %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

# ---------------------------
# 1. Model Initialization (from scratch)
# ---------------------------
# We use the configuration of a popular small-scale LLM (facebook/opt-125m)
# but initialize the model randomly (i.e. train from scratch)
model_name = "facebook/opt-125m"
model_name = "EleutherAI/gpt-neo-125M"
config = AutoConfig.from_pretrained(model_name)  # load config; do NOT load pretrained weights
model = AutoModelForCausalLM.from_config(config)   # randomly initialized model

# Load tokenizer (we can reuse the pretrained tokenizer)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token  # assign PAD token if missing

# Enable gradient checkpointing to reduce memory usage (at the cost of additional compute)
model.gradient_checkpointing_enable()
logger.info(f"Initialized model from scratch with configuration from '{model_name}'.")

# ---------------------------
# 2. Dataset Preparation
# ---------------------------
# Load the Wikitext-2 dataset as our general-purpose text corpus.
# For a quick profiling run, we use only a small subset.
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
train_dataset = dataset["train"].select(range(1000))  # limit to 1000 examples for this demo

# Tokenization: convert text to token IDs (truncated to a maximum length)
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, max_length=128)

tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Data collator: handles padding and prepares labels for causal LM (labels equal to input_ids)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ---------------------------
# 3. Trainer Setup with Memory Profiling Callback
# ---------------------------
# TrainingArguments are set to run only 3 steps and use a small batch size suitable for an 8–12GB GPU.
training_args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=2,
    max_steps=3,  # run only 3 training iterations for profiling
    gradient_accumulation_steps=1,
    fp16=False,  # using full precision; set True if your GPU supports mixed precision to save memory
    logging_steps=1,
    report_to=[],  # disable external logging (e.g., wandb)
    disable_tqdm=False,
    use_cpu=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    callbacks=[ProfilerCallback(), SnapshotCallback()],
)

# ---------------------------
# 4. Training with PyTorch Profiler
# ---------------------------

trainer.train()  # run 3 training iterations

# Print a summary of the profiler's memory usage by CUDA operation (top 10 ops)
print("Profiler Memory Usage Summary (top CUDA ops):")


2025-03-07 20:09:39,876 - INFO - Initialized model from scratch with configuration from 'EleutherAI/gpt-neo-125M'.


Starting profiler...


/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/torch/nn/parallel/data_parallel.py:37: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 1 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
1,5.279300
2,5.157300
3,5.144700


[W307 20:09:47.721222271 CPUAllocator.cpp:245] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event


Stopping profiler...
Profiler Memory Usage Summary (top CUDA ops):


In [8]:
from ures.files import filter_files
from perf_estimator.xmem import XMem
from pathlib import Path


profiler_files = filter_files("pt.trace.json", str(Path().cwd().joinpath("Profiler")), fuzz=True)[-1]

xmen = XMem(
    batch_size=100,
    max_gpu_memory_in_gb=8,
)
result = xmen.estimate(profiler_file=profiler_files, trainer_enable=True)
result


2025-03-06 23:27:22,933 - WARNING - Duplicate layer name found: GPTNeoBlock_11. Renaming to GPTNeoBlock_11_71
2025-03-06 23:27:22,933 - WARNING - Duplicate layer name found: LayerNorm_22. Renaming to LayerNorm_22_39
2025-03-06 23:27:22,933 - WARNING - Duplicate layer name found: GPTNeoAttention_11. Renaming to GPTNeoAttention_11_86
2025-03-06 23:27:22,934 - WARNING - Duplicate layer name found: GPTNeoSelfAttention_11. Renaming to GPTNeoSelfAttention_11_82
2025-03-06 23:27:22,934 - WARNING - Duplicate layer name found: Linear_66. Renaming to Linear_66_fd
2025-03-06 23:27:22,934 - WARNING - Duplicate layer name found: Linear_67. Renaming to Linear_67_38
2025-03-06 23:27:22,934 - WARNING - Duplicate layer name found: Linear_68. Renaming to Linear_68_53
2025-03-06 23:27:22,934 - WARNING - Duplicate layer name found: Dropout_34. Renaming to Dropout_34_7f
2025-03-06 23:27:22,935 - WARNING - Duplicate layer name found: Linear_69. Renaming to Linear_69_97
2025-03-06 23:27:22,935 - WARNING - Du